# Tracking Token Usage and Costs Across Multi-Agent Runs

When running agents in production, understanding how many tokens each agent consumes — and how much that costs — is critical for budgeting and optimization.

The OpenAI Agents SDK exposes a `Usage` object on every run result that tracks:
- `input_tokens` — tokens sent to the model (prompt + history)
- `output_tokens` — tokens generated by the model
- `total_tokens` — sum of both

This notebook shows you how to:
1. Read usage from a simple single-agent run
2. Aggregate usage across a multi-agent pipeline with handoffs
3. Build a reusable cost calculator for OpenAI models
4. Use lifecycle hooks to track usage per-agent in real time
5. Set a token budget and abort when it's exceeded

## Setup

In [ ]:
%pip install openai-agents --quiet

In [ ]:
import asyncio
import os
from dataclasses import dataclass, field
from typing import Any

from agents import (
    Agent,
    RunContextWrapper,
    RunHooks,
    Runner,
    RunResult,
    Tool,
    function_tool,
)
from agents.models.interface import ModelResponse

# Set your API key
os.environ.setdefault("OPENAI_API_KEY", "your-api-key-here")

## Part 1 — Reading Usage from a Single Agent Run

Every `RunResult` exposes a `.usage` attribute with token counts.

In [ ]:
agent = Agent(
    name="Summarizer",
    instructions="Summarize the given text in two sentences.",
    model="gpt-4o-mini",
)

result: RunResult = await Runner.run(
    agent,
    "The OpenAI Agents SDK is a lightweight framework for building multi-agent workflows. "
    "It supports handoffs, tool calling, guardrails, and streaming.",
)

usage = result.usage
print(f"Output : {result.final_output}")
print(f"\nToken usage:")
print(f"  Input tokens  : {usage.input_tokens:,}")
print(f"  Output tokens : {usage.output_tokens:,}")
print(f"  Total tokens  : {usage.total_tokens:,}")

## Part 2 — Cost Calculator for OpenAI Models

Token counts are only useful when you can convert them to dollars. Here's a simple cost calculator using current OpenAI pricing.

In [ ]:
# Pricing per 1M tokens (USD) — update as needed from https://openai.com/pricing
MODEL_PRICING = {
    "gpt-4o": {"input": 2.50, "output": 10.00},
    "gpt-4o-mini": {"input": 0.15, "output": 0.60},
    "gpt-4.1": {"input": 2.00, "output": 8.00},
    "gpt-4.1-mini": {"input": 0.40, "output": 1.60},
    "o3": {"input": 10.00, "output": 40.00},
    "o4-mini": {"input": 1.10, "output": 4.40},
}


def calculate_cost(model: str, input_tokens: int, output_tokens: int) -> float:
    """Return the estimated cost in USD for a model call."""
    if model not in MODEL_PRICING:
        raise ValueError(f"Unknown model: {model}. Add it to MODEL_PRICING.")
    prices = MODEL_PRICING[model]
    return (
        input_tokens * prices["input"] / 1_000_000
        + output_tokens * prices["output"] / 1_000_000
    )


cost = calculate_cost("gpt-4o-mini", usage.input_tokens, usage.output_tokens)
print(f"Estimated cost: ${cost:.6f}")

## Part 3 — Aggregating Usage Across a Multi-Agent Pipeline

In a pipeline where agents hand off to each other, each agent contributes to the total token usage. The SDK accumulates usage across the entire run automatically.

In [ ]:
@function_tool
def get_article_text(topic: str) -> str:
    """Return a short article on the given topic.

    Args:
        topic: The topic to write about.
    """
    return (
        f"Here is a detailed article about {topic}. "
        "It covers the history, current developments, and future outlook. "
        "The field has grown significantly over the past decade, with many "
        "breakthroughs happening in the last few years."
    )


# Agent 1: Fetches and summarizes content
researcher = Agent(
    name="Researcher",
    instructions="Use the get_article_text tool to fetch content, then summarize it in 3 bullet points.",
    tools=[get_article_text],
    model="gpt-4o-mini",
)

# Agent 2: Turns the summary into a polished paragraph
writer = Agent(
    name="Writer",
    instructions="Turn the bullet-point summary into a single polished paragraph suitable for a newsletter.",
    model="gpt-4o-mini",
)

# Run the full pipeline
summary_result = await Runner.run(researcher, "artificial intelligence")
final_result = await Runner.run(writer, summary_result.final_output)

# Aggregate usage manually across both runs
total_input = summary_result.usage.input_tokens + final_result.usage.input_tokens
total_output = summary_result.usage.output_tokens + final_result.usage.output_tokens
total_cost = calculate_cost("gpt-4o-mini", total_input, total_output)

print("Pipeline usage breakdown:")
print(f"  Researcher — input: {summary_result.usage.input_tokens:,}, output: {summary_result.usage.output_tokens:,}")
print(f"  Writer     — input: {final_result.usage.input_tokens:,}, output: {final_result.usage.output_tokens:,}")
print(f"  Total      — input: {total_input:,}, output: {total_output:,}")
print(f"  Estimated cost: ${total_cost:.6f}")
print(f"\nFinal output:\n{final_result.final_output}")

## Part 4 — Live Usage Tracking with Lifecycle Hooks

For long-running agents you may want to observe usage in real time — after each model call — rather than only at the end. Use `RunHooks` for this.

In [ ]:
@dataclass
class UsageAccumulator:
    """Accumulates token usage across all model calls in a run."""
    input_tokens: int = 0
    output_tokens: int = 0
    model_calls: int = 0

    @property
    def total_tokens(self) -> int:
        return self.input_tokens + self.output_tokens

    def record(self, response: ModelResponse) -> None:
        self.input_tokens += response.usage.input_tokens
        self.output_tokens += response.usage.output_tokens
        self.model_calls += 1

    def cost(self, model: str) -> float:
        return calculate_cost(model, self.input_tokens, self.output_tokens)


class UsageTrackingHooks(RunHooks):
    """Lifecycle hooks that record token usage after every model response."""

    def __init__(self, model: str = "gpt-4o-mini") -> None:
        self.model = model
        self.accumulator = UsageAccumulator()

    async def on_model_response(
        self,
        context: RunContextWrapper[Any],
        response: ModelResponse,
    ) -> None:
        self.accumulator.record(response)
        print(
            f"[hook] model call #{self.accumulator.model_calls}: "
            f"+{response.usage.input_tokens} in / "
            f"+{response.usage.output_tokens} out "
            f"(running total: {self.accumulator.total_tokens} tokens, "
            f"~${self.accumulator.cost(self.model):.6f})"
        )


hooks = UsageTrackingHooks(model="gpt-4o-mini")

tracked_result = await Runner.run(
    researcher,
    "quantum computing",
    hooks=hooks,
)

acc = hooks.accumulator
print(f"\nFinal totals:")
print(f"  Model calls   : {acc.model_calls}")
print(f"  Input tokens  : {acc.input_tokens:,}")
print(f"  Output tokens : {acc.output_tokens:,}")
print(f"  Estimated cost: ${acc.cost('gpt-4o-mini'):.6f}")

## Part 5 — Token Budget: Abort When Spending Too Much

For production systems you may want to set a hard token budget and stop the agent if it exceeds it, rather than letting it run indefinitely.

In [ ]:
class BudgetEnforcingHooks(RunHooks):
    """Abort the run if cumulative token usage exceeds a budget."""

    def __init__(self, max_tokens: int, model: str = "gpt-4o-mini") -> None:
        self.max_tokens = max_tokens
        self.model = model
        self.accumulator = UsageAccumulator()

    async def on_model_response(
        self,
        context: RunContextWrapper[Any],
        response: ModelResponse,
    ) -> None:
        self.accumulator.record(response)
        if self.accumulator.total_tokens > self.max_tokens:
            raise RuntimeError(
                f"Token budget exceeded: {self.accumulator.total_tokens:,} tokens used, "
                f"budget was {self.max_tokens:,}. "
                f"Estimated cost so far: ${self.accumulator.cost(self.model):.6f}."
            )


# Try running with a very tight budget to see it trigger
budget_hooks = BudgetEnforcingHooks(max_tokens=50, model="gpt-4o-mini")

simple_agent = Agent(
    name="Simple Agent",
    instructions="Answer in one sentence.",
    model="gpt-4o-mini",
)

try:
    budget_result = await Runner.run(
        simple_agent,
        "Explain what a neural network is.",
        hooks=budget_hooks,
    )
    print(f"Completed within budget. Output: {budget_result.final_output}")
except RuntimeError as exc:
    print(f"Budget exceeded: {exc}")

## Summary

| Technique | When to use |
|---|---|
| `result.usage` | Quick one-off inspection after a run |
| Manual aggregation | Sequential pipeline where you run agents separately |
| `UsageTrackingHooks` | Long runs where you want real-time visibility |
| `BudgetEnforcingHooks` | Production systems with strict cost controls |

### Tips
- Token prices change — keep `MODEL_PRICING` in a config file, not hardcoded.
- `input_tokens` includes the full conversation history on every turn, so costs grow with context length.
- Use `gpt-4o-mini` for early pipeline stages and `gpt-4o` only for final synthesis to minimize costs.
- Combine `BudgetEnforcingHooks` with a fallback to a cheaper model instead of hard-aborting.